# Vectorless RAG 

**Vectorless RAG** is a type of RAG that retrieves information **without using embeddings or a vector database**.

Instead of converting text into vectors and performing similarity search, it uses traditional search techniques such as:

* Keyword matching
* Text search
* BM25 ranking
* SQL queries
* Metadata filtering

---

## How It Works

### 1. Store Documents

Documents are stored in:

* Databases
* Files
* Search indexes

---

### 2. User Asks a Question

Example:

```text
What is LangGraph?
```

---

### 3. Search for Matching Content

The retriever searches using keywords.

Example:

```text
Search: "LangGraph"
```

It finds documents containing that term.

---

### 4. Send Context to LLM

Retrieved text is added to the prompt and sent to the LLM.

---

### 5. Generate Answer

The LLM uses the retrieved information to answer the question.

---

# Flow

```text
Documents
    ↓
Keyword Search / BM25
    ↓
Relevant Text
    ↓
LLM
    ↓
Answer
```

---

# Vectorless RAG vs Traditional RAG

| Vectorless RAG            | Traditional RAG            |
| ------------------------- | -------------------------- |
| Uses keyword search       | Uses embeddings            |
| No vector database needed | Requires vector database   |
| Simple to implement       | More setup required        |
| Faster for exact matches  | Better for semantic search |
| Lower cost                | Higher cost                |

---

# Example

Document:

```text
LangGraph is a framework for building stateful AI agents.
```

User Query:

```text
What is LangGraph?
```

Vectorless RAG:

* Finds the document because it contains the keyword **LangGraph**
* Sends it to the LLM
* LLM generates the answer

---

# Key Idea

**Vectorless RAG = Retrieve documents using keyword/text search instead of embeddings and vector databases, then use an LLM to generate the final answer.**


------------------------------------------------------------------------------------

------------------------------------------------------------------------------------

# PageIndex (Vectorless RAG) 

This diagram shows a **vectorless RAG system** that does not use embeddings or a vector database.

Instead, it builds a **tree structure of the document** and lets the LLM search through that structure.

---

## 1. PDF Document

```text
PDF Document
```

This is the original document (book, report, notes, etc.).

---

## 2. LLM Tree Builder

```text
PDF → LLM Tree Builder
```

The LLM reads the document and organizes it into sections and subsections.

Example:

```text
Document
│
├── Introduction
├── Chapter 1
│   ├── Topic A
│   └── Topic B
├── Chapter 2
└── Conclusion
```

Think of it like creating a **table of contents automatically**.

---

## 3. JSON Tree Index

```text
LLM Tree Builder → JSON Tree Index
```

The hierarchy is stored as JSON.

Example:

```json
{
  "Introduction": {},
  "Chapter 1": {
    "Topic A": {},
    "Topic B": {}
  },
  "Conclusion": {}
}
```

No embeddings are created.
No vector database is needed.

---

## 4. User Query

Example:

```text
What is Topic B about?
```

---

## 5. LLM Tree Search

```text
User Query → LLM Tree Search
```

Instead of searching vectors, the LLM navigates the document tree.

It reasons like:

```text
Question mentions Topic B
↓
Go to Chapter 1
↓
Open Topic B section
↓
Read relevant content
```

---

## 6. Named Sections

After searching the tree, the system doesn't directly send raw document chunks to the LLM.

Instead, it identifies the most relevant sections and creates structured information:

* **Section Title**
* **Page Number**
* **Short Summary**

Example:

```text
Section: Vector Databases
Page: 12
Summary: Explains how embeddings are stored and retrieved.
```

```text
Section: Embedding Models
Page: 15
Summary: Discusses converting text into vectors.
```

### Why is this useful?

* Gives the LLM cleaner context
* Reduces unnecessary text
* Provides page citations
* Makes answers more explainable and traceable

### Final Answer Generation

The LLM then uses these named sections to generate an answer and can cite where the information came from:

```text
According to the "Vector Databases" section (Page 12), embeddings are stored in a vector store for similarity search.
```

### Key Idea

Instead of:

```text
Search → Raw Chunks → LLM
```

PageIndex does:

```text
Search → Named Sections (Title + Page + Summary) → LLM
```

This makes retrieval more structured and human-like.

---

## 7. Retrieve Relevant Sections

The relevant sections are selected from the document.

```text
Topic B Content
```

---

## 8. Generate Final Answer

The retrieved content is given to the LLM.

```text
Relevant Sections
        ↓
       LLM
        ↓
    Final Answer
```

---

# Complete Flow

```text
PDF Document
      ↓
LLM Tree Builder
      ↓
JSON Tree Index
      ↓
User Query
      ↓
LLM Tree Search
      ↓
Relevant Sections
      ↓
LLM
      ↓
Final Answer
```

---

# Key Idea

**Traditional RAG**:

```text
Documents → Embeddings → Vector DB → Retrieval → LLM
```

**PageIndex (Vectorless RAG)**:

```text
Documents → Tree Structure → LLM Search → LLM
```

The system relies on the **document's structure and LLM reasoning** instead of vector similarity search.


------------------------------------------------------------------------------------

------------------------------------------------------------------------------------

This diagram explains **how the JSON Tree Index is built** in a vectorless RAG system.

# Step-by-Step Explanation

### 1. Raw PDF Document

```text
Raw PDF
```

The input can be:

* PDF
* Research paper
* Report
* Any structured document

---

### 2. TOC Detection

```text
Scan first N pages for headings
```

The system first checks whether the document contains a **Table of Contents (TOC)**.

---

### Path A: TOC Exists

```text
Parse existing TOC
```

If a TOC is found:

```text
Chapter 1
Chapter 2
Chapter 3
```

The system extracts the chapter and section structure directly.

This is fast and accurate.

---

### Path B: No TOC

```text
LLM reads pages
```

If no TOC exists:

```text
Page 1
Page 2
Page 3
...
```

The LLM scans pages and tries to infer:

* Headings
* Subheadings
* Section boundaries

Example:

```text
Introduction
Methods
Results
Conclusion
```

This is what the red bracket in your image highlights.

---

### 3. Section-Aware Splitting

```text
Respect logical boundaries
```

Instead of splitting by token count:

❌ Bad

```text
Every 1000 tokens
```

✅ Good

```text
Introduction
Methods
Results
```

Each section remains intact.

---

### 4. LLM Summarizes Each Section

For every section, the LLM generates:

```text
node_id
title
page
summary
```

Example:

```text
node_id: 0006
title: Financial Stability
page: 21
summary: Discusses risks and safeguards in the financial system.
```

---

### 5. Assemble Hierarchical Tree

The sections are organized into a tree.

Example:

```text
Document
│
├── Chapter 1
│   ├── Topic A
│   └── Topic B
│
├── Chapter 2
│   ├── Topic C
│   └── Topic D
```

Relationship:

```text
Parent
 └── Child
      └── Grandchild
```

---

### 6. Output: JSON Tree Index

Final result:

```json
{
  "title": "Financial Stability",
  "node_id": "0006",
  "page": 21,
  "summary": "Discusses risks and safeguards..."
}
```

This tree becomes the searchable index used later by **LLM Tree Search**.

---

# Complete Build Pipeline

```text
Raw PDF
    ↓
TOC Detection
    ↓
 ┌───────────────┬───────────────┐
 │ TOC Exists    │ No TOC        │
 ↓               ↓
Parse TOC    LLM Reads Pages
 └───────────────┘
        ↓
Section-Aware Splitting
        ↓
LLM Summarizes Sections
        ↓
Assemble Hierarchical Tree
        ↓
JSON Tree Index
```

### Key Idea

Instead of creating **embeddings and vectors**, the system creates a **structured tree of sections, titles, page numbers, and summaries**. Later, the LLM navigates this tree to find the most relevant information.


------------------------------------------------------------------------------------

------------------------------------------------------------------------------------

# Retrieval Process in PageIndex (Vectorless RAG)

This diagram shows how the system retrieves information after the JSON Tree Index has already been created.

---

## 1. User Query

The user asks a question.

Example:

```text
What are the risks to financial stability?
```

---

## 2. Read the Tree Index

The LLM first reads the JSON Tree Index.

It scans:

* Titles
* Page numbers
* Summaries

Example:

```text
Financial Stability (Page 21)
Banking Risks (Page 25)
Economic Indicators (Page 30)
```

The LLM gets a high-level overview of the document without reading every page.

---

## 3. Reason and Select Nodes

The LLM thinks about which sections are most relevant.

Example:

```text
Question → Financial Stability
Relevant Nodes:
- Financial Stability
- Banking Risks
```

The system returns a list of selected `node_ids`.

---

## 4. Extract Section Content

Using the selected node IDs, the system fetches the actual content from those pages.

Example:

```text
Node 0006 → Page 21
Node 0008 → Page 25
```

Now the LLM has detailed information instead of just summaries.

---

## 5. Check: Is the Information Sufficient?

The LLM evaluates whether it has enough information to answer the question.

### If YES

Move to answer generation.

### If NO

The system performs a **cross-reference follow-up**.

Example:

```text
See Appendix G
See Related Risks Section
See Chapter 4
```

The LLM navigates to those referenced sections and retrieves additional content.

This creates the **loop-back** shown in the diagram.

---

## 6. Generate Answer

Once enough information is collected:

* The LLM generates the answer.
* Includes citations.

Example:

```text
Financial stability risks include liquidity shortages and
banking failures (Financial Stability, Page 21).
```

---

# Complete Flow

```text
User Query
     ↓
Read Tree Index
     ↓
Reason & Select Nodes
     ↓
Extract Section Content
     ↓
Sufficient Information?
     │
     ├── No
     │      ↓
     │ Cross-Reference Follow-up
     │      ↓
     │ Read More Sections
     │      ↺
     │
     └── Yes
            ↓
     Generate Answer
```

## Key Idea

Unlike traditional RAG, which retrieves chunks using vector similarity, PageIndex uses **LLM reasoning over a document tree**:

```text
Tree Index
    ↓
Reason
    ↓
Select Sections
    ↓
Read Content
    ↓
Answer
```

It behaves more like a human reading a document: first checking the table of contents, then opening relevant sections, following references if needed, and finally answering.
